In [113]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY")
)

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
    model="gemini-2.5-flash"
)

In [5]:
from IPython.display import display, Markdown

In [6]:
Markdown(assistant.rag("How do I run Ollama locally?"))

To run Ollama locally, follow these steps:

1.  **Install Ollama:**
    *   Visit [https://ollama.com/download](https://ollama.com/download).
    *   **macOS**: Download and install the `.pkg` file.
    *   **Windows**: Download and install the `.msi` file.
    *   **Linux**: Open a terminal and run the command:
        ```bash
        curl -fsSL https://ollama.com/install.sh | sh
        ```

2.  **Run a model locally:**
    *   Once installed, open a terminal and type:
        ```bash
        ollama run llama3
        ```
    *   This command will download the LLaMA 3 model (~4GB), start the model locally, and open a chat-like interface for you to type questions.

3.  **Test the Ollama local server (optional):**
    *   In a terminal, run:
        ```bash
        curl http://localhost:11434
        ```
    *   You should receive a response similar to: `{"models": [...]}`

4.  **Install the Python client (optional):**
    *   To interact with Ollama via Python, install the client:
        ```bash
        pip install ollama
        ```
    *   A minimal Python example is:
        ```python
        import ollama

        response = ollama.chat(
            model='llama3',
            messages=[{"role": "user", "content": your_prompt}]
        )

        print(response['message']['content'])
        ```

In [7]:
Markdown(assistant.rag("How do I run Olama locally?"))

I am sorry, but the provided context does not contain information on how to run Olama locally.

In [8]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [52]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
)

response.choices[0].message.content

'That\'s great! To give you the most accurate information, I need a little more detail.\n\nCould you please tell me:\n\n1.  **What is the name of the course?**\n2.  **Where did you discover it (e.g., a specific university, Coursera, edX, Udemy, a private platform, a social media ad, etc.)?**\n\nOnce I have that information, I can help you figure out if you can still join, what the enrollment process is, and any deadlines or prerequisites.\n\nGenerally, here\'s what you\'d typically look for on a course\'s official page:\n\n*   **"Enroll," "Register," or "Sign Up" buttons:** These are usually the direct path to joining.\n*   **Start Dates and Deadlines:** Many courses have specific enrollment periods.\n*   **"About the Course" or "FAQ" sections:** These often detail enrollment procedures, prerequisites, and availability.'

In [53]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [54]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [55]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool],
)

response

ChatCompletion(id='jnw2auWuINvNkdUPzZj5qAU', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-2816617339553363951', function=Function(arguments='{"query":"join course"}', name='search'), type='function')]))], created=1781955727, model='gemini-2.5-flash', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=14, prompt_tokens=63, total_tokens=141, completion_tokens_details=None, prompt_tokens_details=None))

In [56]:
print(response.to_json())

{
  "id": "jnw2auWuINvNkdUPzZj5qAU",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "role": "assistant",
        "tool_calls": [
          {
            "id": "function-call-2816617339553363951",
            "function": {
              "arguments": "{\"query\":\"join course\"}",
              "name": "search"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1781955727,
  "model": "gemini-2.5-flash",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 14,
    "prompt_tokens": 63,
    "total_tokens": 141
  }
}


In [57]:
response.choices[0].message.tool_calls[0].id

'function-call-2816617339553363951'

In [58]:
import json

call = response.choices[0]
args = json.loads(call.message.tool_calls[0].function.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [59]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and Git

In [60]:
messages.append(response.choices[0].message)

messages.append({
    "role": "tool",
    "tool_call_id": call.message.tool_calls[0].id,
    "content": result_json,
})

In [61]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='function-call-2816617339553363951', function=Function(arguments='{"query":"join course"}', name='search'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'function-call-2816617339553363951',
  'content': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n 

In [62]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool],
)

response


ChatCompletion(id='qHw2ary2Nojaxs0P3JrzoAE', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Yes, you can still join the course. However, if you wish to receive a certificate, you'll need to submit your project while submissions are still being accepted.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1781955754, model='gemini-2.5-flash', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=34, prompt_tokens=867, total_tokens=1088, completion_tokens_details=None, prompt_tokens_details=None))

In [63]:
response.usage

CompletionUsage(completion_tokens=34, prompt_tokens=867, total_tokens=1088, completion_tokens_details=None, prompt_tokens_details=None)

# Agentic loop

In [64]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [69]:
def make_call(tool_call):
    args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result_json,
    }

In [73]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool],
)

messages.append(response.choices[0].message)
has_function_calls = False

for tool_call in response.choices[0].message.tool_calls or []:
    if tool_call.type == "function":
        print("function_call:", tool_call.function.name, tool_call.function.arguments)
        call_output = make_call(tool_call)
        messages.append(call_output)
        has_function_calls = True

# Assistant's text is outside the loop
if not has_function_calls:
    print("ASSISTANT:")
    print(response.choices[0].message.content)

function_call: search {"query":"join course"}
function_call: search {"query":"enroll course"}
function_call: search {"query":"late enrollment"}


In [79]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages,
        tools=[search_tool],
    )

    message = response.choices[0].message
    messages.append(message)

    for tool_call in message.tool_calls or []:
        if tool_call.type == "function":
            print("function_call:", tool_call.function.name, tool_call.function.arguments)
            call_output = make_call(tool_call)
            messages.append(call_output)
            has_function_calls = True

    if not has_function_calls:
        print("ASSISTANT:")
        print(message.content)
        break

    it += 1


iteration #1...
function_call: search {"query":"join course"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted.

Would you like to know more about how to get started or the course workflow?


In [84]:
messages[4]

ChatCompletionMessage(content="Yes, you can still join the course! However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted.\n\nWould you like to know more about how to get started or the course workflow?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)

In [95]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches (3 or more).

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=messages,
        tools=[search_tool],
        
    )

    message = response.choices[0].message
    messages.append(message)

    for tool_call in message.tool_calls or []:
        if tool_call.type == "function":
            print("function_call:", tool_call.function.name, tool_call.function.arguments)
            call_output = make_call(tool_call)
            messages.append(call_output)
            has_function_calls = True

    if not has_function_calls:
        print("ASSISTANT:")
        print(message.content)
        break

    it += 1


iteration #1...
function_call: search {"query":"join course"}
function_call: search {"query":"enroll course"}
function_call: search {"query":"course registration"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. However, if you are interested in receiving a certificate, you will need to submit your project while submissions are still being accepted.

Is there anything else you would like to know about the course?


In [98]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

def agent_loop(instructions, question, model="gemini-2.5-flash") -> str:
    messages = [
        {"role": "system", "content": instructions},  # not "developer"
        {"role": "user", "content": question}
    ]

    it = 1
    last_answer = ""

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        message = response.choices[0].message
        messages.append(message)

        for tool_call in message.tool_calls or []:
            if tool_call.type == "function":
                print("function_call:", tool_call.function.name, tool_call.function.arguments)
                call_output = make_call(tool_call)
                messages.append(call_output)
                has_function_calls = True

        if not has_function_calls:
            print("ASSISTANT:")
            last_answer = message.content
            print(last_answer)
            break

        it += 1

    return last_answer

In [99]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"run Olama locally"}
iteration #2...
ASSISTANT:
To run Ollama locally, follow these steps:

1.  **Install Ollama:**
    *   Visit [https://ollama.com/download](https://ollama.com/download) and download the appropriate installer for your operating system (macOS, Windows, or Linux).
    *   For macOS, install the `.pkg` file.
    *   For Windows, install the `.msi` file.
    *   For Linux, run the following command in your terminal:
        ```bash
        curl -fsSL https://ollama.com/install.sh | sh
        ```

2.  **Run a model locally:**
    *   Open a terminal and type:
        ```bash
        ollama run llama3
        ```
        This command will download the LLaMA 3 model (approximately 4GB) and start it locally, opening a chat interface where you can interact with the model.

3.  **Test the Ollama local server:**
    *   In a new terminal, run:
        ```bash
        curl http://localhost:11434
        ```
    *   You should recei

'To run Ollama locally, follow these steps:\n\n1.  **Install Ollama:**\n    *   Visit [https://ollama.com/download](https://ollama.com/download) and download the appropriate installer for your operating system (macOS, Windows, or Linux).\n    *   For macOS, install the `.pkg` file.\n    *   For Windows, install the `.msi` file.\n    *   For Linux, run the following command in your terminal:\n        ```bash\n        curl -fsSL https://ollama.com/install.sh | sh\n        ```\n\n2.  **Run a model locally:**\n    *   Open a terminal and type:\n        ```bash\n        ollama run llama3\n        ```\n        This command will download the LLaMA 3 model (approximately 4GB) and start it locally, opening a chat interface where you can interact with the model.\n\n3.  **Test the Ollama local server:**\n    *   In a new terminal, run:\n        ```bash\n        curl http://localhost:11434\n        ```\n    *   You should receive a JSON response similar to `{"models": [...]}`.\n\n4.  **Install the

In [100]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. However, if you are looking to receive a certificate, you will need to submit your project before the submission window closes.

Is there anything else you would like to know about the course?


'Yes, you can still join the course. However, if you are looking to receive a certificate, you will need to submit your project before the submission window closes.\n\nIs there anything else you would like to know about the course?'

In [102]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches (at least 3). First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course"}
function_call: search {"query":"enrollment deadline"}
function_call: search {"query":"late registration"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! You can start whenever you want, as the videos and GitHub materials are available.

However, if you want to receive a certificate, you'll need to submit your project while submissions are still being accepted and participate in the peer-review process with a "live" cohort. Certificates are not awarded for self-paced mode.

You can find the deadlines on the course management platform. You don't need a confirmation email after registering; you're considered accepted and can just start learning and submitting homework.

Is there anything else you'd like to know about the course?


'Yes, you can still join the course! You can start whenever you want, as the videos and GitHub materials are available.\n\nHowever, if you want to receive a certificate, you\'ll need to submit your project while submissions are still being accepted and participate in the peer-review process with a "live" cohort. Certificates are not awarded for self-paced mode.\n\nYou can find the deadlines on the course management platform. You don\'t need a confirmation email after registering; you\'re considered accepted and can just start learning and submitting homework.\n\nIs there anything else you\'d like to know about the course?'

In [103]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"chess"}
function_call: search {"query":"game theory"}
function_call: search {"query":"popular culture"}
iteration #3...
ASSISTANT:
It seems like "Queen's Gambit" is not directly mentioned in our course FAQ. However, it is a very well-known chess opening and also the title of a popular miniseries.

Would you like to know more about:
1.  **The chess opening:** What it is, its variations, or its strategic implications?
2.  **The miniseries:** What it's about, or its impact on chess popularity?
3.  Something else related to games or strategy?

Let me know what you're most interested in, and I can try to find more information!


'It seems like "Queen\'s Gambit" is not directly mentioned in our course FAQ. However, it is a very well-known chess opening and also the title of a popular miniseries.\n\nWould you like to know more about:\n1.  **The chess opening:** What it is, its variations, or its strategic implications?\n2.  **The miniseries:** What it\'s about, or its impact on chess popularity?\n3.  Something else related to games or strategy?\n\nLet me know what you\'re most interested in, and I can try to find more information!'

In [104]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I can only answer questions about the course or its logistics. I can't find any information about "queen gambit" in the course FAQ, so it's likely an off-topic question. Is there anything else I can help you with regarding the course?


'I can only answer questions about the course or its logistics. I can\'t find any information about "queen gambit" in the course FAQ, so it\'s likely an off-topic question. Is there anything else I can help you with regarding the course?'

# ToyAIKit

In [107]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [108]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [109]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [110]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [111]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [115]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [117]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [118]:
result.cost

CostInfo(input_cost=Decimal('0.00285'), output_cost=Decimal('0.001404'), total_cost=Decimal('0.004254'))

In [1]:
result.all_messages

NameError: name 'result' is not defined

In [120]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


In [121]:
runner.run()

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"course logistics FAQ exam project submission deadline office hours"

In [122]:
!uv add gitsource

Resolved 130 packages in 1.18s                                       
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)-------------------     0 B/8.77 KiB            
⠙ Preparing packages... (0/2)-------------------     0 B/8.77 KiB            
gitsource            ------------------------------     0 B/8.77 KiB
⠙ Preparing packages... (0/2)-------------------     0 B/10.31 KiB           
gitsource            ------------------------------ 8.77 KiB/8.77 KiB
⠙ Preparing packages... (0/2)-------------------     0 B/10.31 KiB           
gitsource            ------------------------------ 8.77 KiB/8.77 KiB
⠙ Preparing packages... (0/2)---------- 10.31 KiB/10.31 KiB         
gitsource            ------------------------------ 8.77 KiB/8.77 KiB
⠙ Preparing packages... (0/2)---------- 10.31 KiB/10.31 KiB         
gitsource            ------------------------------ 8.77 KiB/8.77 KiB
⠙ Preparing packages... (0/2)---------- 10.31 KiB/